# 데이터 불러오기

## 파일 읽으면서 바로 전처리

In [1]:
import pandas as pd
import glob

file_list = glob.glob("Dataset/*.csv")

df_list = []

for file in file_list:
    df = pd.read_csv(file, encoding="cp949")

    # 컬럼 정리
    df.columns = df.columns.str.strip()
    df.columns = df.columns.str.replace(".1", "", regex=False)

    # 시간 변환
    if "센서 시간" in df.columns:
        df["센서 시간"] = pd.to_datetime(
            df["센서 시간"].str.replace("_", " "),
            errors="coerce"
        )

    df_list.append(df)

df_all = pd.concat(df_list, axis=0, ignore_index=True)

df_all.shape

C:\Users\hoons\AppData\Local\Temp\ipykernel_7224\1096873460.py:9: DtypeWarning: Columns (53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding="cp949")
C:\Users\hoons\AppData\Local\Temp\ipykernel_7224\1096873460.py:9: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding="cp949")


(3979057, 59)

# 데이터 중간 저장

In [2]:
df_all.to_csv(r"C:\Users\hoons\OneDrive\바탕 화면\휴먼 프로젝트 2\df_all.csv", index=False, encoding="utf-8-sig")

In [7]:
df = pd.read_csv(r"C:\Users\hoons\OneDrive\바탕 화면\휴먼 프로젝트 2\df_all.csv", encoding="utf-8-sig")

C:\Users\hoons\AppData\Local\Temp\ipykernel_7224\1442999869.py:1: DtypeWarning: Columns (12,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\hoons\OneDrive\바탕 화면\휴먼 프로젝트 2\df_all.csv", encoding="utf-8-sig")


# 데이터 전처리

## 정렬

In [8]:
df = df.sort_values(
    ["시리얼", "센서 시간"]
).reset_index(drop=True)

In [9]:
df.shape

(3979057, 59)

## 시간 칼럼 정리

In [10]:
df["센서 시간"] = pd.to_datetime(
    df["센서 시간"].str.replace("_", " "),
    errors="coerce"
)

df["데이터수집시간"] = pd.to_datetime(
    df["데이터수집시간"],
    errors="coerce"
)

df = df.sort_values(["시리얼", "센서 시간"]).reset_index(drop=True)

df[["시리얼", "센서 시간", "자치구역", "행정구역"]].head()

,시리얼,센서 시간,자치구역,행정구역
0,OC3CL200010,2025-11-12 20:07:00,Seoul_Grand_Park,meeting_bridge2
1,OC3CL200010,2025-11-12 21:07:00,Seoul_Grand_Park,meeting_bridge2
2,OC3CL200010,2025-11-12 22:07:00,Seoul_Grand_Park,meeting_bridge2
3,OC3CL200010,2025-11-12 23:07:00,Seoul_Grand_Park,meeting_bridge2
4,OC3CL200010,2025-11-13 00:07:00,Seoul_Grand_Park,meeting_bridge2


## 결측률 확인

In [11]:
missing_ratio = (
    df.isna().mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_ratio.columns = ["컬럼명", "결측률"]

missing_ratio["결측률(%)"] = (missing_ratio["결측률"] * 100).round(2)
missing_ratio

,컬럼명,결측률,결측률(%)
0,최대 흑구온도,1.000000,100.00
1,평균 흑구온도.1,1.000000,100.00
2,평균 흑구온도,1.000000,100.00
3,최소 오존,0.964986,96.50
4,최대 오존,0.964972,96.50
5,평균 오존,0.964929,96.49
6,최대 이산화황,0.947474,94.75
7,최소 이산화황,0.947283,94.73
8,평균 이산화황,0.947202,94.72
9,최소 황화수소,0.946047,94.60


## 결측률 90% 이상 컬럼 제거

In [12]:
threshold = 0.9

drop_cols = missing_ratio.loc[
    missing_ratio["결측률"] >= threshold,
    "컬럼명"
].tolist()

drop_cols

['최대 흑구온도',
 '평균 흑구온도.1',
 '평균 흑구온도',
 '최소 오존',
 '최대 오존',
 '평균 오존',
 '최대 이산화황',
 '최소 이산화황',
 '평균 이산화황',
 '최소 황화수소',
 '최소 암모니아',
 '최대 황화수소',
 '최대 암모니아',
 '평균 황화수소',
 '평균 암모니아',
 '최소 풍속.1',
 '최소 풍속',
 '최대 풍속',
 '최대 풍향',
 '평균 풍속',
 '평균 풍속.1',
 '최소 이산화질소',
 '최소 일산화탄소',
 '평균 이산화질소',
 '최대 일산화탄소',
 '최대 이산화질소',
 '평균 일산화탄소']

In [13]:
df_clean = df.drop(columns=drop_cols)

df_clean.shape

(3979057, 32)

## 중복 컬럼 정리

In [14]:
duplicated_like_cols = [col for col in df_clean.columns if ".1" in col]
duplicated_like_cols

[]

In [15]:
df_clean = df_clean.drop(columns=duplicated_like_cols, errors="ignore")

## 숫자형 칼럼만 따로 뽑기

In [17]:
id_cols = [
    "모델명", "시리얼", "센서 시간", 
    "지역구분", "자치구역", "행정구역",
    "데이터수집시간", "데이터구분번호"
]

numeric_cols = [
    col for col in df_clean.columns
    if col not in id_cols
]

numeric_cols

['최대 기온',
 '평균 기온',
 '최소 기온',
 '최대 상대습도',
 '평균 상대습도',
 '최소 상대습도',
 '최대 조도',
 '평균 조도',
 '최소 조도',
 '최대 자외선',
 '평균 자외선',
 '최소 자외선',
 '최대 소음',
 '평균 소음',
 '최소 소음',
 '최대 진동(x)',
 '평균 진동(x)',
 '최소 진동(x)',
 '최대 진동(y)',
 '평균 진동(y)',
 '최소 진동(y)',
 '최대 진동(z)',
 '평균 진동(z)',
 '최소 진동(z)']

## 이상값 처리

In [18]:
import numpy as np

range_rules = {
    "평균 기온": (-30, 50),
    "최대 기온": (-30, 60),
    "최소 기온": (-40, 50),
    "평균 상대습도": (0, 100),
    "최대 상대습도": (0, 100),
    "최소 상대습도": (0, 100),
    "평균 조도": (0, 200000),
    "최대 조도": (0, 200000),
    "최소 조도": (0, 200000),
    "평균 자외선": (0, 20),
    "최대 자외선": (0, 20),
    "최소 자외선": (0, 20),
    "평균 소음": (0, 120),
    "최대 소음": (0, 140),
    "최소 소음": (0, 120),
}

for col, (low, high) in range_rules.items():
    if col in df_clean.columns:
        df_clean.loc[
            (df_clean[col] < low) | (df_clean[col] > high),
            col
        ] = np.nan

## 센서별 시간 보간

In [19]:
df_clean = df_clean.sort_values(["시리얼", "센서 시간"])

df_clean[numeric_cols] = (
    df_clean
    .groupby("시리얼")[numeric_cols]
    .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

## 남은 결측치 처리

In [20]:
for col in numeric_cols:
    if col not in df_clean.columns:
        continue

    # 행정구역 기준 중앙값
    df_clean[col] = df_clean[col].fillna(
        df_clean.groupby("행정구역")[col].transform("median")
    )

    # 자치구역 기준 중앙값
    df_clean[col] = df_clean[col].fillna(
        df_clean.groupby("자치구역")[col].transform("median")
    )

    # 전체 중앙값
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

## 시간 파생변수 만들기

In [21]:
df_clean["hour"] = df_clean["센서 시간"].dt.hour
df_clean["day"] = df_clean["센서 시간"].dt.day
df_clean["dayofweek"] = df_clean["센서 시간"].dt.dayofweek
df_clean["is_weekend"] = df_clean["dayofweek"].isin([5, 6]).astype(int)

## 최종 결측치 확인
### 센서 시간 없는 행 제거

In [24]:
df_clean = df_clean.dropna(
    subset=["센서 시간", "지역구분", "자치구역", "행정구역"]
).copy()

df_clean["hour"] = df_clean["센서 시간"].dt.hour
df_clean["day"] = df_clean["센서 시간"].dt.day
df_clean["dayofweek"] = df_clean["센서 시간"].dt.dayofweek
df_clean["is_weekend"] = df_clean["dayofweek"].isin([5, 6]).astype(int)

df_clean = df_clean.reset_index(drop=True)

In [25]:
missing_ratio = (
    df_clean.isna().mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_ratio.columns = ["컬럼명", "결측률"]

missing_ratio["결측률(%)"] = (missing_ratio["결측률"] * 100).round(2)
missing_ratio

,컬럼명,결측률,결측률(%)
0,시리얼,0.0,0.0
1,센서 시간,0.0,0.0
2,지역구분,0.0,0.0
3,자치구역,0.0,0.0
4,행정구역,0.0,0.0
5,최대 기온,0.0,0.0
6,평균 기온,0.0,0.0
7,최소 기온,0.0,0.0
8,최대 상대습도,0.0,0.0
9,평균 상대습도,0.0,0.0


## 모델명, 데이터 구분번호 칼럼 제거

In [23]:
drop_cols = ["데이터구분번호", "모델명"]

df_clean = df_clean.drop(columns=[col for col in drop_cols if col in df_clean.columns])

## 모델에 쓸 핵심 컬럼만 저장

In [ ]:
model_base_cols = [
    "시리얼", "센서 시간", "지역구분", "자치구역", "행정구역",
    "hour", "day", "dayofweek", "is_weekend",
    "평균 기온", "평균 상대습도", "평균 조도", "평균 자외선",
    "평균 소음"
]

model_base_cols = [col for col in model_base_cols if col in df_clean.columns]

df_model_base = df_clean[model_base_cols].copy()

df_model_base.head()

In [ ]:
df_model_base.to_csv(
    "S_DOT_ENV_clean_base.csv",
    index=False,
    encoding="utf-8-sig"
)

# 모델별 데이터셋 나누기
## 쾌적도 모델용

In [ ]:
comfort_cols = [
    "시리얼", "센서 시간", "자치구역", "행정구역",
    "hour", "dayofweek",
    "평균 기온", "평균 상대습도", "평균 조도",
    "평균 자외선", "평균 소음"
]

comfort_cols = [col for col in comfort_cols if col in df_clean.columns]

df_comfort = df_clean[comfort_cols].copy()

## 건강 위험도 모델용

In [ ]:
health_cols = [
    "시리얼", "센서 시간", "자치구역", "행정구역",
    "hour", "dayofweek",
    "평균 기온", "평균 상대습도",
    "평균 자외선", "평균 소음"
]

health_cols = [col for col in health_cols if col in df_clean.columns]

df_health = df_clean[health_cols].copy()

## 소음/진동 스트레스 모델용
- 진동 컬럼이 90% 이상 결측이면 제거됐을 가능성이 크다. 그러면 소음 중심 모델

In [ ]:
stress_cols = [
    "시리얼", "센서 시간", "자치구역", "행정구역",
    "hour", "dayofweek",
    "평균 소음", "최대 소음", "최소 소음",
    "평균 진동(x)", "평균 진동(y)", "평균 진동(z)"
]

stress_cols = [col for col in stress_cols if col in df_clean.columns]

df_stress = df_clean[stress_cols].copy()

## 냉난방 필요도 모델용

In [ ]:
energy_cols = [
    "시리얼", "센서 시간", "자치구역", "행정구역",
    "hour", "dayofweek",
    "평균 기온", "최대 기온", "최소 기온",
    "평균 상대습도", "평균 조도"
]

energy_cols = [col for col in energy_cols if col in df_clean.columns]

df_energy = df_clean[energy_cols].copy()

## 저장

In [ ]:
df_comfort.to_csv("comfort_dataset.csv", index=False, encoding="utf-8-sig")
df_health.to_csv("health_dataset.csv", index=False, encoding="utf-8-sig")
df_stress.to_csv("stress_dataset.csv", index=False, encoding="utf-8-sig")
df_energy.to_csv("energy_dataset.csv", index=False, encoding="utf-8-sig")